In [ ]:
# =============================================================================
# CLO-SKET PHASE CONDITIONING
# CELL 1 — LOAD FROZEN OBJECTS + DISCOVER AVAILABLE PREDICTION LINEAGE
# =============================================================================

import pickle
import numpy as np
from pathlib import Path

print("=" * 92)
print("CLO-SKET — PHASE CONDITIONING")
print("CELL 1 — LOAD FROZEN OBJECTS + DISCOVER AVAILABLE PREDICTION LINEAGE")
print("=" * 92)

OUT_DIR = Path("/content/drive/MyDrive/FashionAI")
PKL_PATH = OUT_DIR / "CLO_SKET_FINAL_IDENTITY_FIGURES.pkl"

if not PKL_PATH.exists():
    raise FileNotFoundError(
        f"Frozen final object file not found:\n{PKL_PATH}"
    )

with open(PKL_PATH, "rb") as f:
    frozen = pickle.load(f)

if not isinstance(frozen, dict):
    raise TypeError(
        f"Expected dictionary, got {type(frozen)}"
    )

print(f"\nLoaded: {PKL_PATH}")
print(f"Objects available: {len(frozen)}")

# -------------------------------------------------------------------------
# Complete object inventory
# -------------------------------------------------------------------------

print("\nFROZEN OBJECT INVENTORY")
print("-" * 92)

for i, name in enumerate(sorted(frozen.keys()), start=1):

    obj = frozen[name]

    try:
        arr = np.asarray(obj)

        if arr.dtype != object:
            detail = (
                f"shape={arr.shape}, "
                f"dtype={arr.dtype}"
            )
        else:
            detail = f"type={type(obj).__name__}"

    except Exception:
        detail = f"type={type(obj).__name__}"

    print(
        f"{i:2d}. {name:40s} {detail}"
    )

# -------------------------------------------------------------------------
# Search specifically for prediction-like objects
# -------------------------------------------------------------------------

prediction_tokens = [
    "hat",
    "pred",
    "oof",
    "learn",
    "recon",
    "c2",
    "s2",
    "r2",
    "mu2",
]

candidate_names = []

for name in frozen.keys():

    lower = name.lower()

    if any(
        token in lower
        for token in prediction_tokens
    ):
        candidate_names.append(name)

print("\nPREDICTION / HARMONIC CANDIDATES")
print("-" * 92)

for name in sorted(candidate_names):

    obj = frozen[name]

    try:
        arr = np.asarray(obj)

        print(
            f"{name:40s} "
            f"shape={arr.shape}, "
            f"dtype={arr.dtype}"
        )

    except Exception:

        print(
            f"{name:40s} "
            f"type={type(obj).__name__}"
        )

# -------------------------------------------------------------------------
# Required observed objects only
# -------------------------------------------------------------------------

observed_required = [
    "garment_identity_ids",
    "cell30m_fold_assignment",
    "C2_obs",
    "S2_obs",
    "R2_obs",
    "mu2_obs_deg",
]

missing_observed = [
    name
    for name in observed_required
    if name not in frozen
]

if missing_observed:

    raise RuntimeError(
        "Missing observed objects: "
        + ", ".join(missing_observed)
    )

# -------------------------------------------------------------------------
# Load observed field
# -------------------------------------------------------------------------

garment_identity_ids = np.asarray(
    frozen["garment_identity_ids"]
).copy()

cell30m_fold_assignment = np.asarray(
    frozen["cell30m_fold_assignment"]
).copy()

C2_obs = np.asarray(
    frozen["C2_obs"],
    dtype=float,
).copy()

S2_obs = np.asarray(
    frozen["S2_obs"],
    dtype=float,
).copy()

R2_obs = np.asarray(
    frozen["R2_obs"],
    dtype=float,
).copy()

mu2_obs_deg = np.asarray(
    frozen["mu2_obs_deg"],
    dtype=float,
).copy()

expected_shape = (2300, 25)

for name, arr in {
    "C2_obs": C2_obs,
    "S2_obs": S2_obs,
    "R2_obs": R2_obs,
    "mu2_obs_deg": mu2_obs_deg,
}.items():

    assert arr.shape == expected_shape
    assert np.isfinite(arr).all()

# -------------------------------------------------------------------------
# Observed harmonic identity
# -------------------------------------------------------------------------

max_R_identity_error = float(
    np.max(
        np.abs(
            R2_obs
            -
            np.sqrt(
                C2_obs**2
                + S2_obs**2
            )
        )
    )
)

assert max_R_identity_error < 1e-12

print("\nOBSERVED FIELD AUDIT")
print("-" * 92)
print(f"Sketches                         : {R2_obs.shape[0]}")
print(f"Radial shells                    : {R2_obs.shape[1]}")
print(
    f"Garment identities               : "
    f"{len(np.unique(garment_identity_ids))}"
)
print(
    f"Locked folds                     : "
    f"{len(np.unique(cell30m_fold_assignment))}"
)
print(
    f"max |R2 - sqrt(C2²+S2²)|         : "
    f"{max_R_identity_error:.3e}"
)

print("\nSTATUS")
print("-" * 92)

if all(
    name in frozen
    for name in [
        "C2_hat",
        "S2_hat",
        "R2_hat",
        "mu2_hat_deg",
    ]
):
    print(
        "Direct predicted harmonic fields are present."
    )
else:
    print(
        "Direct C2_hat/S2_hat/R2_hat/mu2_hat_deg fields "
        "are NOT present in this frozen file."
    )
    print(
        "Prediction lineage must be identified before "
        "phase-conditioning analysis proceeds."
    )

print(
    "\nPASS — observed harmonic field loaded; "
    "prediction-lineage discovery completed."
)
print(
    "No model was fitted and no primary-analysis object was modified."
)

CLO-SKET — PHASE CONDITIONING
CELL 1 — LOAD FROZEN OBJECTS + DISCOVER AVAILABLE PREDICTION LINEAGE

Loaded: /content/drive/MyDrive/FashionAI/CLO_SKET_FINAL_IDENTITY_FIGURES.pkl
Objects available: 35

FROZEN OBJECT INVENTORY
--------------------------------------------------------------------------------------------
 1. C2_hat_identity_oof_field                shape=(2300, 25), dtype=float64
 2. C2_obs                                   shape=(2300, 25), dtype=float64
 3. R2_hat_identity_oof_field                shape=(2300, 25), dtype=float64
 4. R2_obs                                   shape=(2300, 25), dtype=float64
 5. S2_hat_identity_oof_field                shape=(2300, 25), dtype=float64
 6. S2_obs                                   shape=(2300, 25), dtype=float64
 7. __NOTE__                                 shape=(), dtype=<U150
 8. boot_reconstruction_30o                  shape=(5000, 9), dtype=float64
 9. categories_identity_30o                  type=ndarray
10. cell30m_fold_ass

In [ ]:
# =============================================================================
# CLO-SKET PHASE CONDITIONING
# CELL 2 — LOAD FROZEN OOF PREDICTION FIELDS + DERIVE CONDITIONING QUANTITIES
# =============================================================================

import numpy as np
import pandas as pd

print("=" * 92)
print("CLO-SKET — PHASE CONDITIONING")
print("CELL 2 — LOAD FROZEN OOF PREDICTIONS + DERIVE CONDITIONING QUANTITIES")
print("=" * 92)

# -------------------------------------------------------------------------
# 1. Load exact frozen identity-disjoint OOF prediction fields
# -------------------------------------------------------------------------

C2_hat = np.asarray(
    frozen["C2_hat_identity_oof_field"],
    dtype=float,
).copy()

S2_hat = np.asarray(
    frozen["S2_hat_identity_oof_field"],
    dtype=float,
).copy()

R2_hat = np.asarray(
    frozen["R2_hat_identity_oof_field"],
    dtype=float,
).copy()

mu2_hat_deg = np.asarray(
    frozen["mu2_hat_identity_oof_deg_field"],
    dtype=float,
).copy()

expected_shape = (2300, 25)

for name, arr in {
    "C2_hat": C2_hat,
    "S2_hat": S2_hat,
    "R2_hat": R2_hat,
    "mu2_hat_deg": mu2_hat_deg,
}.items():

    assert arr.shape == expected_shape
    assert np.isfinite(arr).all()

# -------------------------------------------------------------------------
# 2. Internal prediction identity
# -------------------------------------------------------------------------

R2_hat_check = np.sqrt(
    C2_hat**2 + S2_hat**2
)

max_hat_identity_error = float(
    np.max(
        np.abs(
            R2_hat_check - R2_hat
        )
    )
)

assert max_hat_identity_error < 1e-12

# -------------------------------------------------------------------------
# 3. Axial distance helper
# -------------------------------------------------------------------------

def axial_distance_deg(a, b):

    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    d = np.mod(
        np.abs(a - b),
        180.0,
    )

    return np.minimum(
        d,
        180.0 - d,
    )

# -------------------------------------------------------------------------
# 4. OOF Cartesian perturbations
# -------------------------------------------------------------------------

dC = C2_hat - C2_obs
dS = S2_hat - S2_obs

component_error_norm = np.sqrt(
    dC**2 + dS**2
)

axial_error_deg = axial_distance_deg(
    mu2_obs_deg,
    mu2_hat_deg,
)

# -------------------------------------------------------------------------
# 5. First-order phase-conditioning quantities
#
# mu = 1/2 atan2(S, C)
#
# dmu ~= (C dS - S dC) / (2 R^2)
#
# and
#
# |dmu| <= ||d(C,S)|| / (2R)
#
# -------------------------------------------------------------------------

R_EPS = 1e-12

safe_R = np.maximum(
    R2_obs,
    R_EPS,
)

phase_linearized_rad = (
    C2_obs * dS
    -
    S2_obs * dC
) / (
    2.0 * safe_R**2
)

phase_linearized_abs_deg = np.abs(
    np.degrees(
        phase_linearized_rad
    )
)

phase_bound_rad = (
    component_error_norm
    /
    (2.0 * safe_R)
)

phase_bound_deg = np.degrees(
    phase_bound_rad
)

inverse_R = 1.0 / safe_R

# -------------------------------------------------------------------------
# 6. Peak-shell quantities
# -------------------------------------------------------------------------

peak_shell_idx = np.asarray(
    frozen["cell30n_peak_shell_index"],
    dtype=int,
)

assert peak_shell_idx.shape == (2300,)

row_idx = np.arange(
    2300
)

R2_peak_obs = R2_obs[
    row_idx,
    peak_shell_idx,
]

R2_peak_hat = R2_hat[
    row_idx,
    peak_shell_idx,
]

mu2_peak_obs_deg = mu2_obs_deg[
    row_idx,
    peak_shell_idx,
]

mu2_peak_hat_deg = mu2_hat_deg[
    row_idx,
    peak_shell_idx,
]

peak_axial_error_deg = axial_distance_deg(
    mu2_peak_obs_deg,
    mu2_peak_hat_deg,
)

peak_component_error_norm = (
    component_error_norm[
        row_idx,
        peak_shell_idx,
    ]
)

peak_phase_linearized_abs_deg = (
    phase_linearized_abs_deg[
        row_idx,
        peak_shell_idx,
    ]
)

peak_phase_bound_deg = (
    phase_bound_deg[
        row_idx,
        peak_shell_idx,
    ]
)

peak_inverse_R = (
    inverse_R[
        row_idx,
        peak_shell_idx,
    ]
)

# -------------------------------------------------------------------------
# 7. Cross-check against frozen Cell 30N peak objects
# -------------------------------------------------------------------------

frozen_peak_R2_obs = np.asarray(
    frozen["cell30n_observed_peak_R2"],
    dtype=float,
)

frozen_peak_mu_obs = np.asarray(
    frozen["cell30n_observed_peak_mu2_deg"],
    dtype=float,
)

frozen_peak_mu_hat = np.asarray(
    frozen["cell30n_identity_oof_peak_mu2_deg"],
    dtype=float,
)

frozen_peak_error = np.asarray(
    frozen["cell30n_identity_oof_peak_axial_error_deg"],
    dtype=float,
)

assert np.allclose(
    R2_peak_obs,
    frozen_peak_R2_obs,
    atol=1e-12,
    rtol=0.0,
)

assert np.allclose(
    mu2_peak_obs_deg,
    frozen_peak_mu_obs,
    atol=1e-12,
    rtol=0.0,
)

assert np.allclose(
    mu2_peak_hat_deg,
    frozen_peak_mu_hat,
    atol=1e-12,
    rtol=0.0,
)

assert np.allclose(
    peak_axial_error_deg,
    frozen_peak_error,
    atol=1e-12,
    rtol=0.0,
)

# -------------------------------------------------------------------------
# 8. Descriptive audit
# -------------------------------------------------------------------------

summary_df = pd.DataFrame({
    "quantity": [
        "Observed R2",
        "OOF component error norm",
        "Actual axial error (deg)",
        "Linearized axial error (deg)",
        "Cauchy phase bound (deg)",
        "1 / R2",
    ],
    "median": [
        np.median(R2_obs),
        np.median(component_error_norm),
        np.median(axial_error_deg),
        np.median(phase_linearized_abs_deg),
        np.median(phase_bound_deg),
        np.median(inverse_R),
    ],
    "mean": [
        np.mean(R2_obs),
        np.mean(component_error_norm),
        np.mean(axial_error_deg),
        np.mean(phase_linearized_abs_deg),
        np.mean(phase_bound_deg),
        np.mean(inverse_R),
    ],
    "p95": [
        np.quantile(R2_obs, 0.95),
        np.quantile(component_error_norm, 0.95),
        np.quantile(axial_error_deg, 0.95),
        np.quantile(phase_linearized_abs_deg, 0.95),
        np.quantile(phase_bound_deg, 0.95),
        np.quantile(inverse_R, 0.95),
    ],
})

print("\nPREDICTION FIELD AUDIT")
print("-" * 92)

print(
    f"max |R2_hat - sqrt(Chat²+Shat²)| : "
    f"{max_hat_identity_error:.3e}"
)

print("\nWHOLE-FIELD CONDITIONING SUMMARY")
print("-" * 92)

print(
    summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("\nPEAK-SHELL SUMMARY")
print("-" * 92)

print(
    f"Median observed peak R2         : "
    f"{np.median(R2_peak_obs):.6f}"
)

print(
    f"Median component-error norm     : "
    f"{np.median(peak_component_error_norm):.6f}"
)

print(
    f"Median actual axial error       : "
    f"{np.median(peak_axial_error_deg):.6f} deg"
)

print(
    f"Median linearized axial error   : "
    f"{np.median(peak_phase_linearized_abs_deg):.6f} deg"
)

print(
    f"Median Cauchy phase bound       : "
    f"{np.median(peak_phase_bound_deg):.6f} deg"
)

print("\nPASS — exact frozen OOF prediction fields loaded.")
print(
    "Phase-conditioning quantities derived without refitting any model."
)

CLO-SKET — PHASE CONDITIONING
CELL 2 — LOAD FROZEN OOF PREDICTIONS + DERIVE CONDITIONING QUANTITIES

PREDICTION FIELD AUDIT
--------------------------------------------------------------------------------------------
max |R2_hat - sqrt(Chat²+Shat²)| : 1.110e-16

WHOLE-FIELD CONDITIONING SUMMARY
--------------------------------------------------------------------------------------------
                    quantity    median      mean       p95
                 Observed R2  0.318041  0.335334  0.681295
    OOF component error norm  0.149973  0.189943  0.497978
    Actual axial error (deg) 10.204589 25.942298 85.552535
Linearized axial error (deg)  2.681628  7.539019 14.140753
    Cauchy phase bound (deg) 21.071413 26.046479 44.416917
                      1 / R2  3.144249 33.719910 26.456819

PEAK-SHELL SUMMARY
--------------------------------------------------------------------------------------------
Median observed peak R2         : 0.660428
Median component-error norm     : 0.168318

In [ ]:
# =============================================================================
# CLO-SKET PHASE CONDITIONING
# CELL 3R — CORRECT GARMENT-LEVEL RECIPROCAL CHECK
# =============================================================================

print("=" * 92)
print("CLO-SKET — PHASE CONDITIONING")
print("CELL 3R — CORRECT GARMENT-LEVEL RECIPROCAL CHECK")
print("=" * 92)

rho_inverse = float(
    identity_assoc_df.loc[
        identity_assoc_df["analysis"]
        ==
        "Median peak 1/R2 vs median axial error",
        "spearman_rho",
    ].iloc[0]
)

rho_R = float(
    identity_assoc_df.loc[
        identity_assoc_df["analysis"]
        ==
        "Median peak R2 vs median axial error",
        "spearman_rho",
    ].iloc[0]
)

print("\nGARMENT-LEVEL RECIPROCAL ASSOCIATION")
print("-" * 92)

print(
    f"rho(median R2, axial error)     : {rho_R:+.6f}"
)

print(
    f"rho(median 1/R2, axial error)   : {rho_inverse:+.6f}"
)

print(
    f"sum                            : "
    f"{rho_R + rho_inverse:+.6f}"
)

print("""
NOTE:
At sketch/shell level, 1/R2 is a strictly decreasing transform of R2,
so Spearman correlations reverse sign exactly.

After aggregation by garment identity, however,

    median(1/R2) != 1 / median(R2)

in general. Therefore exact sign reversal is not mathematically required.
The near-equal magnitudes observed here are nevertheless consistent with
the expected inverse relationship.
""")

assert rho_R < 0
assert rho_inverse > 0

assert abs(
    abs(rho_R) - abs(rho_inverse)
) < 0.01

print("PASS — garment-level reciprocal relationship verified correctly.")

CLO-SKET — PHASE CONDITIONING
CELL 3R — CORRECT GARMENT-LEVEL RECIPROCAL CHECK

GARMENT-LEVEL RECIPROCAL ASSOCIATION
--------------------------------------------------------------------------------------------
rho(median R2, axial error)     : -0.355875
rho(median 1/R2, axial error)   : +0.355135
sum                            : -0.000740

NOTE:
At sketch/shell level, 1/R2 is a strictly decreasing transform of R2,
so Spearman correlations reverse sign exactly.

After aggregation by garment identity, however,

    median(1/R2) != 1 / median(R2)

in general. Therefore exact sign reversal is not mathematically required.
The near-equal magnitudes observed here are nevertheless consistent with
the expected inverse relationship.

PASS — garment-level reciprocal relationship verified correctly.


In [ ]:
# =============================================================================
# CLO-SKET PHASE CONDITIONING
# CELL 4 — MAGNITUDE-STRATIFIED PHASE-CONDITIONING ANALYSIS
# =============================================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

print("=" * 92)
print("CLO-SKET — PHASE CONDITIONING")
print("CELL 4 — MAGNITUDE-STRATIFIED PHASE-CONDITIONING ANALYSIS")
print("=" * 92)

# -------------------------------------------------------------------------
# 1. Work at garment-identity level
#
# Stratification is performed using each garment identity's median
# observed peak R2. This preserves the independent analysis unit.
#
# Quartiles are descriptive conditioning strata, not inferential groups.
# -------------------------------------------------------------------------

phase_strat_df = identity_phase_df.copy()

phase_strat_df["R2_quartile"] = pd.qcut(
    phase_strat_df["median_peak_R2"],
    q=4,
    labels=[
        "Q1 weakest",
        "Q2",
        "Q3",
        "Q4 strongest",
    ],
)

# -------------------------------------------------------------------------
# 2. Quartile boundaries
# -------------------------------------------------------------------------

q_edges = np.quantile(
    phase_strat_df["median_peak_R2"],
    [0.0, 0.25, 0.50, 0.75, 1.0],
)

print("\nOBSERVED PEAK-R2 QUARTILE BOUNDARIES")
print("-" * 92)

for i in range(4):
    print(
        f"Q{i+1}: "
        f"{q_edges[i]:.6f} -> "
        f"{q_edges[i+1]:.6f}"
    )

# -------------------------------------------------------------------------
# 3. Descriptive summaries by harmonic-strength quartile
# -------------------------------------------------------------------------

strat_rows = []

for quartile, sub in phase_strat_df.groupby(
    "R2_quartile",
    observed=True,
):

    strat_rows.append({
        "R2_quartile": str(quartile),
        "n_identities": len(sub),

        "median_peak_R2":
            float(
                np.median(
                    sub["median_peak_R2"]
                )
            ),

        "median_component_error_norm":
            float(
                np.median(
                    sub[
                        "median_peak_component_error_norm"
                    ]
                )
            ),

        "median_conditioning_bound_deg":
            float(
                np.median(
                    sub[
                        "median_peak_conditioning_bound_deg"
                    ]
                )
            ),

        "median_linearized_error_deg":
            float(
                np.median(
                    sub[
                        "median_peak_linearized_error_deg"
                    ]
                )
            ),

        "median_actual_axial_error_deg":
            float(
                np.median(
                    sub[
                        "median_peak_axial_error_deg"
                    ]
                )
            ),

        "mean_actual_axial_error_deg":
            float(
                np.mean(
                    sub[
                        "median_peak_axial_error_deg"
                    ]
                )
            ),

        "p90_actual_axial_error_deg":
            float(
                np.quantile(
                    sub[
                        "median_peak_axial_error_deg"
                    ],
                    0.90,
                )
            ),
    })

phase_strat_summary_df = pd.DataFrame(
    strat_rows
)

print("\nGARMENT-IDENTITY CONDITIONING BY OBSERVED PEAK-R2 QUARTILE")
print("-" * 92)

print(
    phase_strat_summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

# -------------------------------------------------------------------------
# 4. Error-threshold proportions
#
# These make the conditioning pattern intuitive:
#
#   <= 5 deg
#   <= 15 deg
#   > 30 deg
#   > 45 deg
# -------------------------------------------------------------------------

threshold_rows = []

for quartile, sub in phase_strat_df.groupby(
    "R2_quartile",
    observed=True,
):

    err = np.asarray(
        sub["median_peak_axial_error_deg"],
        dtype=float,
    )

    threshold_rows.append({
        "R2_quartile": str(quartile),
        "n_identities": len(err),
        "prop_error_le_5deg":
            float(np.mean(err <= 5.0)),
        "prop_error_le_15deg":
            float(np.mean(err <= 15.0)),
        "prop_error_gt_30deg":
            float(np.mean(err > 30.0)),
        "prop_error_gt_45deg":
            float(np.mean(err > 45.0)),
    })

phase_error_threshold_df = pd.DataFrame(
    threshold_rows
)

print("\nAXIAL-ERROR THRESHOLD PROPORTIONS")
print("-" * 92)

print(
    phase_error_threshold_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

# -------------------------------------------------------------------------
# 5. Weakest vs strongest harmonic regime
# -------------------------------------------------------------------------

weak = phase_strat_df[
    phase_strat_df["R2_quartile"]
    == "Q1 weakest"
]

strong = phase_strat_df[
    phase_strat_df["R2_quartile"]
    == "Q4 strongest"
]

weak_error = float(
    np.median(
        weak["median_peak_axial_error_deg"]
    )
)

strong_error = float(
    np.median(
        strong["median_peak_axial_error_deg"]
    )
)

weak_bound = float(
    np.median(
        weak["median_peak_conditioning_bound_deg"]
    )
)

strong_bound = float(
    np.median(
        strong["median_peak_conditioning_bound_deg"]
    )
)

weak_component = float(
    np.median(
        weak["median_peak_component_error_norm"]
    )
)

strong_component = float(
    np.median(
        strong["median_peak_component_error_norm"]
    )
)

print("\nWEAKEST VS STRONGEST HARMONIC REGIME")
print("-" * 92)

print(
    f"Median actual axial error      : "
    f"{weak_error:.6f} -> {strong_error:.6f} deg"
)

print(
    f"Median conditioning bound      : "
    f"{weak_bound:.6f} -> {strong_bound:.6f} deg"
)

print(
    f"Median component-error norm    : "
    f"{weak_component:.6f} -> {strong_component:.6f}"
)

if strong_error > 0:
    print(
        f"Weak/strong axial-error ratio  : "
        f"{weak_error / strong_error:.6f}"
    )

# -------------------------------------------------------------------------
# 6. Ordinal quartile trend
#
# This is descriptive. Quartile number is used only to summarize
# monotonic ordering across harmonic-strength strata.
# -------------------------------------------------------------------------

quartile_code = (
    phase_strat_df["R2_quartile"]
    .cat.codes
    .to_numpy()
    + 1
)

rho_q_error, p_q_error = spearmanr(
    quartile_code,
    phase_strat_df[
        "median_peak_axial_error_deg"
    ],
)

rho_q_bound, p_q_bound = spearmanr(
    quartile_code,
    phase_strat_df[
        "median_peak_conditioning_bound_deg"
    ],
)

rho_q_component, p_q_component = spearmanr(
    quartile_code,
    phase_strat_df[
        "median_peak_component_error_norm"
    ],
)

print("\nDESCRIPTIVE ORDINAL QUARTILE TRENDS")
print("-" * 92)

print(
    f"Quartile strength vs axial error       : "
    f"rho={rho_q_error:+.6f}"
)

print(
    f"Quartile strength vs conditioning bound: "
    f"rho={rho_q_bound:+.6f}"
)

print(
    f"Quartile strength vs component error   : "
    f"rho={rho_q_component:+.6f}"
)

# -------------------------------------------------------------------------
# 7. Check whether the conditioning-bound pattern is monotonic
# -------------------------------------------------------------------------

bound_by_q = (
    phase_strat_summary_df[
        "median_conditioning_bound_deg"
    ].to_numpy()
)

error_by_q = (
    phase_strat_summary_df[
        "median_actual_axial_error_deg"
    ].to_numpy()
)

bound_monotone_decreasing = bool(
    np.all(
        np.diff(bound_by_q) <= 0
    )
)

error_monotone_decreasing = bool(
    np.all(
        np.diff(error_by_q) <= 0
    )
)

print("\nMONOTONICITY AUDIT")
print("-" * 92)

print(
    f"Conditioning bound decreases Q1 -> Q4 : "
    f"{bound_monotone_decreasing}"
)

print(
    f"Actual axial error decreases Q1 -> Q4 : "
    f"{error_monotone_decreasing}"
)

# -------------------------------------------------------------------------
# 8. Interpretation
# -------------------------------------------------------------------------

print("\n" + "=" * 92)
print("MAGNITUDE-STRATIFIED INTERPRETATION")
print("=" * 92)

print("""
Observed harmonic magnitude defines the conditioning of phase recovery:
for a fixed Cartesian perturbation, shorter second-harmonic vectors
produce larger angular perturbations.

This stratified analysis is descriptive and does not treat quartiles
as independent experimental groups. Its purpose is to determine whether
the empirical OOF reconstruction follows the qualitative ordering
predicted by the phase-conditioning geometry.

Component reconstruction error and harmonic magnitude may both vary
between strata. Therefore any decline in angular error with increasing
R2 should not be interpreted as a causal effect of R2 alone.
""")

print("PASS — magnitude-stratified phase-conditioning analysis completed.")

CLO-SKET — PHASE CONDITIONING
CELL 4 — MAGNITUDE-STRATIFIED PHASE-CONDITIONING ANALYSIS

OBSERVED PEAK-R2 QUARTILE BOUNDARIES
--------------------------------------------------------------------------------------------
Q1: 0.324453 -> 0.609565
Q2: 0.609565 -> 0.661108
Q3: 0.661108 -> 0.703177
Q4: 0.703177 -> 0.809339

GARMENT-IDENTITY CONDITIONING BY OBSERVED PEAK-R2 QUARTILE
--------------------------------------------------------------------------------------------
 R2_quartile  n_identities  median_peak_R2  median_component_error_norm  median_conditioning_bound_deg  median_linearized_error_deg  median_actual_axial_error_deg  mean_actual_axial_error_deg  p90_actual_axial_error_deg
  Q1 weakest            58        0.579599                     0.202780                      10.160330                     3.268987                       5.988056                    14.218851                   42.844573
          Q2            57        0.638629                     0.168084                 

In [ ]:
# =============================================================================
# CLO-SKET PHASE CONDITIONING
# CELL 5 — FINAL PHASE-CONDITIONING DIAGNOSTIC LOCK
# =============================================================================

import numpy as np
import pandas as pd

print("=" * 92)
print("CLO-SKET — PHASE CONDITIONING")
print("CELL 5 — FINAL PHASE-CONDITIONING DIAGNOSTIC LOCK")
print("=" * 92)

# -------------------------------------------------------------------------
# 1. Recover locked garment-level association values from Cell 3
# -------------------------------------------------------------------------

def get_identity_rho(label):
    return float(
        identity_assoc_df.loc[
            identity_assoc_df["analysis"] == label,
            "spearman_rho",
        ].iloc[0]
    )

rho_R2 = get_identity_rho(
    "Median peak R2 vs median axial error"
)

rho_inverse_R2 = get_identity_rho(
    "Median peak 1/R2 vs median axial error"
)

rho_component = get_identity_rho(
    "Median peak component-error norm vs median axial error"
)

rho_bound = get_identity_rho(
    "Median conditioning bound vs median axial error"
)

rho_linearized = get_identity_rho(
    "Median linearized error vs median actual axial error"
)

# -------------------------------------------------------------------------
# 2. Recover locked quartile evidence from Cell 4
# -------------------------------------------------------------------------

weak = phase_strat_summary_df.loc[
    phase_strat_summary_df["R2_quartile"] == "Q1 weakest"
].iloc[0]

strong = phase_strat_summary_df.loc[
    phase_strat_summary_df["R2_quartile"] == "Q4 strongest"
].iloc[0]

weak_R2 = float(weak["median_peak_R2"])
strong_R2 = float(strong["median_peak_R2"])

weak_error = float(
    weak["median_actual_axial_error_deg"]
)

strong_error = float(
    strong["median_actual_axial_error_deg"]
)

weak_bound = float(
    weak["median_conditioning_bound_deg"]
)

strong_bound = float(
    strong["median_conditioning_bound_deg"]
)

weak_component = float(
    weak["median_component_error_norm"]
)

strong_component = float(
    strong["median_component_error_norm"]
)

weak_strong_error_ratio = (
    weak_error / strong_error
)

# -------------------------------------------------------------------------
# 3. Manuscript-facing numerical lock
# -------------------------------------------------------------------------

phase_conditioning_lock_df = pd.DataFrame([
    {
        "evidence":
            "Observed peak R2 vs axial error",
        "analysis_unit":
            "230 garment identities",
        "result":
            f"Spearman rho = {rho_R2:+.3f}",
        "interpretation":
            "Stronger harmonic magnitude is associated with lower axial error",
    },
    {
        "evidence":
            "Peak Cartesian error vs axial error",
        "analysis_unit":
            "230 garment identities",
        "result":
            f"Spearman rho = {rho_component:+.3f}",
        "interpretation":
            "Cartesian reconstruction perturbation strongly tracks angular error",
    },
    {
        "evidence":
            "Conditioning bound vs axial error",
        "analysis_unit":
            "230 garment identities",
        "result":
            f"Spearman rho = {rho_bound:+.3f}",
        "interpretation":
            "Combined perturbation/magnitude conditioning quantity tracks axial error strongly",
    },
    {
        "evidence":
            "Linearized phase error vs actual error",
        "analysis_unit":
            "230 garment identities",
        "result":
            f"Spearman rho = {rho_linearized:+.3f}",
        "interpretation":
            "First-order phase perturbation is strongly consistent with observed error",
    },
    {
        "evidence":
            "Weakest vs strongest R2 quartile",
        "analysis_unit":
            "Garment-identity quartiles",
        "result":
            (
                f"median axial error "
                f"{weak_error:.3f}° -> {strong_error:.3f}°; "
                f"ratio = {weak_strong_error_ratio:.2f}x"
            ),
        "interpretation":
            "Weak-harmonic identities show approximately twice the median axial error",
    },
    {
        "evidence":
            "Weakest vs strongest conditioning bound",
        "analysis_unit":
            "Garment-identity quartiles",
        "result":
            (
                f"{weak_bound:.3f}° -> "
                f"{strong_bound:.3f}°"
            ),
        "interpretation":
            "Empirical strata follow the direction predicted by conditioning geometry",
    },
])

print("\nMANUSCRIPT-FACING PHASE-CONDITIONING SUMMARY")
print("-" * 92)

print(
    phase_conditioning_lock_df.to_string(
        index=False
    )
)

# -------------------------------------------------------------------------
# 4. Hard provenance / consistency guards
# -------------------------------------------------------------------------

# Frozen Cell 30N peak result must remain reproduced.
assert abs(
    np.median(peak_axial_error_deg)
    - 4.104118
) < 1e-5

# Expected direction of R2 relationship.
assert rho_R2 < 0
assert rho_inverse_R2 > 0

# Cartesian perturbation must positively track angular error.
assert rho_component > 0

# The theoretically motivated combined conditioning quantity must
# positively track actual axial error.
assert rho_bound > 0

# First-order approximation must also track actual error positively.
assert rho_linearized > 0

# In these frozen results the combined conditioning quantity is more
# strongly associated with error than R2 magnitude alone.
assert abs(rho_bound) > abs(rho_R2)

# Quartile ordering established descriptively in Cell 4.
assert weak_R2 < strong_R2
assert weak_error > strong_error
assert weak_bound > strong_bound

# -------------------------------------------------------------------------
# 5. Mathematical claim lock
# -------------------------------------------------------------------------

print("\n" + "=" * 92)
print("FINAL PHASE-CONDITIONING INTERPRETATION LOCK")
print("=" * 92)

print(r"""
1. GEOMETRIC BASIS

   For the axial second harmonic

       F2 = C2 - i S2 = R2 exp(-i 2 mu2),

   orientation is

       mu2 = 1/2 atan2(S2, C2).

   Its first-order perturbation is

       dmu2
         = (C2 dS2 - S2 dC2) / (2 R2^2),

   with the bound

       |dmu2|
         <= ||d(C2,S2)|| / (2 R2).

   Phase estimation is therefore intrinsically less well-conditioned
   as harmonic magnitude R2 approaches zero.


2. EMPIRICAL CONSISTENCY

   At the garment-identity level, observed peak R2 was negatively
   associated with axial reconstruction error.

   Cartesian reconstruction perturbation was substantially more strongly
   associated with axial error, and the combined conditioning quantity

       ||Delta(C2,S2)|| / (2 R2)

   showed the strongest of these associations with observed axial error.


3. MAGNITUDE STRATIFICATION

   Across increasing observed peak-R2 quartiles, both the median
   conditioning bound and median actual axial error decreased
   monotonically.

   The weakest-harmonic quartile showed approximately twice the median
   axial error of the strongest-harmonic quartile.


4. INTERPRETATION

   The negative R2-error association should therefore not be treated
   as an isolated empirical phenomenon. It is consistent with the
   expected geometry of phase estimation.

   However, R2 alone does not determine reconstruction error.
   Cartesian prediction perturbation also varies across observations
   and contributes strongly to angular error.


5. CLAIM BOUNDARY

   These results demonstrate consistency with phase-conditioning
   geometry; they do not establish that R2 causally determines error,
   that the first-order approximation is exact for large perturbations,
   or that all empirical angular error is explained by conditioning.


6. PRIMARY ANALYSIS STATUS

   No model was refitted, no fold assignment was changed, and no
   primary-analysis object was modified. All conditioning analyses use
   the frozen identity-disjoint OOF reconstruction fields.
""")

print("PASS — Point 4 phase conditioning scientifically locked.")

CLO-SKET — PHASE CONDITIONING
CELL 5 — FINAL PHASE-CONDITIONING DIAGNOSTIC LOCK

MANUSCRIPT-FACING PHASE-CONDITIONING SUMMARY
--------------------------------------------------------------------------------------------
                               evidence              analysis_unit                                             result                                                                    interpretation
        Observed peak R2 vs axial error     230 garment identities                              Spearman rho = -0.356                  Stronger harmonic magnitude is associated with lower axial error
    Peak Cartesian error vs axial error     230 garment identities                              Spearman rho = +0.760               Cartesian reconstruction perturbation strongly tracks angular error
      Conditioning bound vs axial error     230 garment identities                              Spearman rho = +0.789 Combined perturbation/magnitude conditioning quantity tracks ax